In [8]:
import pandas as pd
import glob
import os
from tqdm import tqdm
# ---------------------------------------------------------------
# CONFIGURACIÓN INICIAL
# ---------------------------------------------------------------
ruta_raw = "../data/raw"
ruta_output = "../data/clean"
os.makedirs(ruta_output, exist_ok=True)

# Buscar todos los CSV
archivos = sorted(glob.glob(os.path.join(ruta_raw, "emergencias_*.csv")))
archivos = archivos[:3]
print(f"Archivos encontrados: {len(archivos)}\n")

Archivos encontrados: 3



In [6]:
# ---------------------------------------------------------------
# FUNCIÓN DE LIMPIEZA POR ARCHIVO
# ---------------------------------------------------------------
def cargar_y_limpiar(path):
    df = pd.read_csv(path, sep=';', encoding='utf-8', dayfirst=True, on_bad_lines='skip', low_memory=False)
    
    # Normalizar nombres de columnas
    df.columns = (
        df.columns.str.strip()
        .str.replace(" ", "_")
        .str.replace("-", "_")
        .str.lower()
    )

    # Asegurar existencia de columnas clave
    expected_cols = ['fecha','provincia','canton','cod_parroquia','parroquia','servicio','subtipo']
    for col in expected_cols:
        if col not in df.columns:
            df[col] = None

    # Estandarizar texto (mayúsculas -> minúsculas -> formato título)
    text_cols = ['provincia','canton','parroquia','servicio','subtipo']
    for c in text_cols:
        df[c] = (
            df[c].astype(str)
            .str.strip()
            .str.title()
            .replace('Nan', pd.NA)
        )

    # Convertir cod_parroquia a texto
    df['cod_parroquia'] = df['cod_parroquia'].astype(str).str.replace('.0','', regex=False)

    # Convertir fecha a datetime
    df['fecha'] = pd.to_datetime(df['fecha'], errors='coerce', dayfirst=True)

    # Eliminar filas con datos faltantes críticos
    df = df.dropna(subset=['fecha','provincia','servicio'])

    # Quitar registros de "Zona No Delimitada"
    df = df[df['provincia'] != 'Zona No Delimitada']

    cols_group = ['fecha','provincia','canton','cod_parroquia','parroquia','servicio','subtipo']
    df_grouped = (
        df.groupby(cols_group)
          .size()
          .reset_index(name='cantidad_eventos')
    )
    
    return df_grouped



In [9]:
dfs = []
for archivo in tqdm(archivos, desc="Procesando archivos ECU911", ncols=80):
    df_clean = cargar_y_limpiar(archivo)
    dfs.append(df_clean)

data = pd.concat(dfs, ignore_index=True)
print("Total de filas después de limpieza básica:", len(data))


Procesando archivos ECU911: 100%|█████████████████| 3/3 [00:05<00:00,  1.83s/it]

Total de filas después de limpieza básica: 301982


In [10]:
data

,fecha,provincia,canton,cod_parroquia,parroquia,servicio,subtipo,cantidad_eventos
0,2022-04-01,Azuay,Camilo Ponce Enriquez,11550,"Camilo Ponce Enríquez, Cabecera Cantonal",Gestión Sanitaria,Problemas En El Embarazo,1
1,2022-04-01,Azuay,Camilo Ponce Enriquez,11550,"Camilo Ponce Enríquez, Cabecera Cantonal",Seguridad Ciudadana,Patrullaje Policial En El Sector Solicitado,1
2,2022-04-01,Azuay,Camilo Ponce Enriquez,11550,"Camilo Ponce Enríquez, Cabecera Cantonal",Seguridad Ciudadana,Resguardo A Personas Que Traslada Valores A Ni...,1
3,2022-04-01,Azuay,Chordeleg,11150,"Chordeleg, Cabecera Cantonal",Seguridad Ciudadana,Patrullaje Policial En El Sector Solicitado,1
4,2022-04-01,Azuay,Cuenca,10150,"Cuenca, Cabecera Cantonal Y Capital Provincial.",Gestión De Riesgos,Amenazas Naturales,1
...,...,...,...,...,...,...,...,...
301977,2021-08-31,Zamora Chinchipe,Zamora,190151,Cumbaratza,Seguridad Ciudadana,Escándalo En Espacio Privado,1
301978,2021-08-31,Zamora Chinchipe,Zamora,190151,Cumbaratza,Seguridad Ciudadana,Libadores,1
301979,2021-08-31,Zamora Chinchipe,Zamora,190151,Cumbaratza,Seguridad Ciudadana,Patrullaje Policial En El Sector Solicitado,1
301980,2021-08-31,Zamora Chinchipe,Zamora,190158,San Carlos De Las Minas,Gestión Sanitaria,"Colisión, Choque Y/O Volcamiento",1


In [12]:
path_out = os.path.join(ruta_output, "ecu911_emergencias_test.parquet")
data.to_parquet(path_out, index=False)
print(f"Archivo limpio guardado en: {path_out}")

Archivo limpio guardado en: ../data/clean/ecu911_emergencias_test.parquet


In [13]:
# (opcional) verificar
check = pd.read_parquet(path_out)
print("Registros verificados:", len(check))

Registros verificados: 301982


In [15]:
check.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 301982 entries, 0 to 301981
Data columns (total 8 columns):
 #   Column            Non-Null Count   Dtype         
---  ------            --------------   -----         
 0   fecha             301982 non-null  datetime64[ns]
 1   provincia         301982 non-null  object        
 2   canton            301982 non-null  object        
 3   cod_parroquia     301982 non-null  object        
 4   parroquia         301982 non-null  object        
 5   servicio          301982 non-null  object        
 6   subtipo           301982 non-null  object        
 7   cantidad_eventos  301982 non-null  int64         
dtypes: datetime64[ns](1), int64(1), object(6)
memory usage: 18.4+ MB


In [16]:
fecha_min = check['fecha'].min().strftime('%d/%m/%Y')
fecha_max = check['fecha'].max().strftime('%d/%m/%Y')
print(f"Rango de fechas: {fecha_min} → {fecha_max}")


Rango de fechas: 01/08/2021 → 30/04/2023
